# Three-Layer GCN (Proposed) vs. Two-Layer GCN (Baseline) on Cora

**Task:** Semi-supervised node classification with Vanilla GCN (Kipf & Welling, 2017).

| Role | Model |
|---|---|
| Baseline | 2-layer GCN (`in → 16 → classes`) |
| **Proposed** | **3-layer GCN (`in → 16 → 16 → classes`)** |

**Notebook outline**
1. Mathematical derivation (spectral convolution → GCN propagation rule → 2-layer and 3-layer forward passes)
2. Implementation from scratch (PyTorch; PyG is used only to download the dataset)
3. Multi-seed experiments (10 seeds) + statistical testing
4. Training curves, depth ablation, over-smoothing analysis, t-SNE
5. Results discussion

> Runtime → Change runtime type → GPU (optional; Cora is small and also runs on CPU).

## 1. Mathematical Derivation

### 1.1 Setup
Graph $G=(V,E)$ with $N=|V|$ nodes, adjacency $A\in\mathbb{R}^{N\times N}$, degree matrix $D_{ii}=\sum_j A_{ij}$, node features $X\in\mathbb{R}^{N\times F}$. Only a small labelled subset $\mathcal{Y}_L\subset V$ is used for training.

### 1.2 Spectral graph convolution
The normalized Laplacian is $L = I_N - D^{-1/2}AD^{-1/2} = U\Lambda U^\top$. The graph Fourier transform of a signal $x$ is $U^\top x$, so a spectral filter $g_\theta(\Lambda)$ acts as

$$
g_\theta \star x = U\, g_\theta(\Lambda)\, U^\top x .
$$

This needs an $O(N^2)$ multiplication and an eigendecomposition, which is expensive.

### 1.3 Chebyshev approximation
Approximate $g_\theta(\Lambda)$ by a truncated Chebyshev expansion of order $K$:

$$
g_{\theta'}(\Lambda) \approx \sum_{k=0}^{K}\theta'_k\,T_k(\tilde\Lambda),\qquad \tilde\Lambda=\tfrac{2}{\lambda_{\max}}\Lambda - I_N ,
$$

with $T_0=1,\;T_1(x)=x,\;T_k(x)=2xT_{k-1}(x)-T_{k-2}(x)$. This gives

$$
g_{\theta'}\star x \approx \sum_{k=0}^{K}\theta'_k\,T_k(\tilde L)\,x,\qquad \tilde L=\tfrac{2}{\lambda_{\max}}L-I_N,
$$

which is $K$-localized (depends only on nodes within $K$ hops).

### 1.4 First-order simplification ($K=1$, $\lambda_{\max}\approx 2$)
$$
g_{\theta'}\star x \approx \theta'_0 x + \theta'_1 (L-I_N)x = \theta'_0 x - \theta'_1 D^{-1/2}AD^{-1/2}x .
$$
Tying parameters $\theta=\theta'_0=-\theta'_1$ gives

$$
g_\theta\star x \approx \theta\left(I_N + D^{-1/2}AD^{-1/2}\right)x .
$$

The eigenvalues of $I_N + D^{-1/2}AD^{-1/2}$ lie in $[0,2]$, so stacking many layers can cause exploding or vanishing gradients.

### 1.5 Renormalization trick
Replace $I_N + D^{-1/2}AD^{-1/2}$ with

$$
\boxed{\hat A=\tilde D^{-1/2}\tilde A\tilde D^{-1/2}},\qquad \tilde A=A+I_N,\quad \tilde D_{ii}=\sum_j\tilde A_{ij}.
$$

Self-loops let a node keep its own information, and the symmetric normalization keeps feature scales stable. Generalizing to $C$ input channels and $F'$ filters:

$$
\boxed{H^{(\ell+1)}=\sigma\!\left(\hat A\,H^{(\ell)}W^{(\ell)}\right)},\qquad H^{(0)}=X,\; W^{(\ell)}\in\mathbb{R}^{d_\ell\times d_{\ell+1}} .
$$

Complexity per layer is $O(|E|\,d_\ell\,d_{\ell+1})$ using sparse $\hat A$.

### 1.6 Baseline: two-layer GCN
$$
Z_{2L}=\mathrm{softmax}\!\Big(\hat A\;\mathrm{ReLU}\big(\hat A X W^{(0)}\big)\,W^{(1)}\Big),\quad W^{(0)}\in\mathbb{R}^{F\times h},\;W^{(1)}\in\mathbb{R}^{h\times C}
$$
Receptive field: **2 hops**. Parameters: $Fh + h + hC + C$.

### 1.7 Proposed: three-layer GCN
$$
Z_{3L}=\mathrm{softmax}\!\Big(\hat A\;\mathrm{ReLU}\Big(\hat A\;\mathrm{ReLU}\big(\hat A X W^{(0)}\big)W^{(1)}\Big)W^{(2)}\Big),
$$
$$
W^{(0)}\in\mathbb{R}^{F\times h},\;W^{(1)}\in\mathbb{R}^{h\times h},\;W^{(2)}\in\mathbb{R}^{h\times C}.
$$
Receptive field: **3 hops** (equivalent to $\hat A^3$ propagation). Parameters: $Fh + h + h^2 + h + hC + C$, which is only $h^2+h$ more than the baseline.

**Hypothesis.** A third layer lets each node aggregate information from 3-hop neighbours, which may help nodes in sparsely labelled regions. However, additional smoothing and extra parameters could also cause over-smoothing or overfitting given only 140 labelled nodes. The experiments below test this empirically.

### 1.8 Training objective
Cross-entropy over labelled nodes with $L_2$ regularization on the first-layer weights (as in Kipf & Welling):

$$
\mathcal{L}=-\sum_{i\in\mathcal{Y}_L}\sum_{c=1}^{C}Y_{ic}\ln Z_{ic}\;+\;\lambda\,\lVert W^{(0)}\rVert_F^2 .
$$

**Gradient of the last layer.** Let $S=\hat A H^{(L-1)}W^{(L-1)}$ (logits). With softmax and cross-entropy, $\frac{\partial\mathcal{L}}{\partial S}=M\odot(Z-Y)$ where $M$ masks the labelled rows. Then

$$
\frac{\partial\mathcal{L}}{\partial W^{(L-1)}}=(\hat A H^{(L-1)})^\top\,\frac{\partial\mathcal{L}}{\partial S},\qquad
\frac{\partial\mathcal{L}}{\partial H^{(L-1)}}=\hat A^\top\,\frac{\partial\mathcal{L}}{\partial S}\,W^{(L-1)\top}.
$$

The backward pass therefore multiplies by $\hat A^\top=\hat A$ once per layer: gradients flow through $L$ propagation steps, which is why depth interacts with smoothing.

### 1.9 Over-smoothing
Repeated multiplication by $\hat A$ drives features toward the dominant eigenvector of $\hat A$ (eigenvalue 1), i.e. $\hat A^k X\to$ a vector proportional to $\tilde D^{1/2}\mathbf{1}$ for each connected component. We measure this with the **mean pairwise cosine similarity** of hidden node embeddings (higher means more over-smoothed).

In [ ]:
!pip -q install torch_geometric

In [ ]:
import time, copy, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.manifold import TSNE
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| torch", torch.__version__)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

## 2. Data
Cora: 2,708 papers, 5,429 citation links, 1,433 bag-of-words features, 7 classes. Standard Planetoid split: 140 train / 500 validation / 1,000 test nodes. Features are row-normalized.

In [ ]:
dataset = Planetoid(root="data/Planetoid", name="Cora", transform=T.NormalizeFeatures())
data = dataset[0].to(device)
N, Fdim, C = data.num_nodes, dataset.num_features, dataset.num_classes
print(data)
print(f"Nodes={N}, Edges(directed)={data.edge_index.size(1)}, Features={Fdim}, Classes={C}")
print(f"Train/Val/Test = {int(data.train_mask.sum())}/{int(data.val_mask.sum())}/{int(data.test_mask.sum())}")

## 3. Implementation

### 3.1 Renormalized adjacency $\hat A=\tilde D^{-1/2}(A+I)\tilde D^{-1/2}$

In [ ]:
def build_norm_adj(edge_index, n):
    loops = torch.arange(n, device=edge_index.device)
    ei = torch.cat([edge_index, torch.stack([loops, loops])], dim=1)       # A + I
    A = torch.sparse_coo_tensor(ei, torch.ones(ei.size(1), device=ei.device), (n, n)).coalesce()
    idx = A.indices()
    A = torch.sparse_coo_tensor(idx, torch.ones(idx.size(1), device=ei.device), (n, n)).coalesce()  # binarize duplicates
    idx = A.indices()
    deg = torch.sparse.sum(A, dim=1).to_dense()                           # D~
    dinv = deg.pow(-0.5)
    vals = dinv[idx[0]] * dinv[idx[1]]                                    # D~^-1/2 A~ D~^-1/2
    return torch.sparse_coo_tensor(idx, vals, (n, n)).coalesce()

A_hat = build_norm_adj(data.edge_index, N)

# sanity checks: symmetry and spectral radius <= 1
dense = A_hat.to_dense()
print("Symmetric:", torch.allclose(dense, dense.T, atol=1e-6))
print("Largest eigenvalue (should be 1.0):", torch.linalg.eigvalsh(dense.cpu()).max().item())
del dense

### 3.2 GCN layer and configurable-depth model
`num_layers=2` is the baseline; `num_layers=3` is the proposed model. Everything else (hidden size, dropout, optimizer, epochs) is identical, so depth is the only variable.

In [ ]:
class GCNLayer(nn.Module):
    """H' = A_hat @ (H W) + b   (activation applied outside)"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Parameter(torch.empty(in_dim, out_dim))
        self.b = nn.Parameter(torch.zeros(out_dim))
        nn.init.xavier_uniform_(self.W)                                    # Glorot init

    def forward(self, x, A):
        return torch.sparse.mm(A, x @ self.W) + self.b

class GCN(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim, num_layers=2, dropout=0.5):
        super().__init__()
        assert num_layers >= 1
        dims = [in_dim] + [hid_dim] * (num_layers - 1) + [out_dim]
        self.layers = nn.ModuleList([GCNLayer(dims[i], dims[i+1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, A, return_hidden=False):
        hidden = []
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, self.dropout, self.training)
            x = layer(x, A)
            if i < len(self.layers) - 1:
                x = F.relu(x)
                hidden.append(x)
        return (x, hidden) if return_hidden else x                         # logits (softmax is inside the loss)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

for L in (2, 3):
    print(f"{L}-layer GCN parameters:", count_params(GCN(Fdim, 16, C, L)))

### 3.3 Training and evaluation
Hyperparameters follow Kipf & Welling: hidden = 16, dropout = 0.5, Adam (lr = 0.01), weight decay 5e-4 on the **first layer only**, 200 epochs. The checkpoint with the best validation accuracy (ties broken by lower validation loss) is used for testing.

In [ ]:
CFG = dict(hidden=16, dropout=0.5, lr=0.01, wd=5e-4, epochs=200)

@torch.no_grad()
def evaluate(model, mask):
    model.eval()
    out = model(data.x, A_hat)
    loss = F.cross_entropy(out[mask], data.y[mask]).item()
    acc = (out[mask].argmax(1) == data.y[mask]).float().mean().item()
    return loss, acc

def train_model(num_layers, seed, cfg=CFG, verbose=False):
    set_seed(seed)
    model = GCN(Fdim, cfg["hidden"], C, num_layers, cfg["dropout"]).to(device)
    opt = torch.optim.Adam([
        dict(params=model.layers[0].parameters(), weight_decay=cfg["wd"]),
        dict(params=[p for l in model.layers[1:] for p in l.parameters()], weight_decay=0.0),
    ], lr=cfg["lr"])

    hist = dict(train_loss=[], val_loss=[], train_acc=[], val_acc=[])
    best = dict(val_acc=-1, val_loss=1e9, state=None, epoch=0)
    t0 = time.time()
    for ep in range(1, cfg["epochs"] + 1):
        model.train(); opt.zero_grad()
        out = model(data.x, A_hat)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward(); opt.step()

        _, tr_acc = evaluate(model, data.train_mask)
        va_loss, va_acc = evaluate(model, data.val_mask)
        hist["train_loss"].append(loss.item()); hist["val_loss"].append(va_loss)
        hist["train_acc"].append(tr_acc);       hist["val_acc"].append(va_acc)
        if va_acc > best["val_acc"] or (va_acc == best["val_acc"] and va_loss < best["val_loss"]):
            best.update(val_acc=va_acc, val_loss=va_loss, epoch=ep, state=copy.deepcopy(model.state_dict()))
        if verbose and ep % 20 == 0:
            print(f"ep {ep:3d} | loss {loss.item():.4f} | train {tr_acc:.3f} | val {va_acc:.3f}")
    train_time = time.time() - t0

    model.load_state_dict(best["state"])
    te_loss, te_acc = evaluate(model, data.test_mask)
    return dict(model=model, hist=hist, test_acc=te_acc, test_loss=te_loss, val_acc=best["val_acc"],
                best_epoch=best["epoch"], train_time=train_time, params=count_params(model))

# quick smoke test
r = train_model(3, seed=0, verbose=True)
print(f"3-layer smoke test -> test acc {r['test_acc']:.4f}")

## 4. Main Experiment: Baseline vs. Proposed (10 seeds)
Each model is trained with 10 different random seeds (weight initialization and dropout). We report mean ± std.

In [ ]:
SEEDS = list(range(10))
MODELS = {"Baseline: 2-layer GCN": 2, "Proposed: 3-layer GCN": 3}
runs = {name: [train_model(L, s) for s in SEEDS] for name, L in MODELS.items()}

rows = []
for name, rs in runs.items():
    te = np.array([r["test_acc"] for r in rs]) * 100
    va = np.array([r["val_acc"] for r in rs]) * 100
    rows.append({
        "Model": name,
        "Params": rs[0]["params"],
        "Val Acc (%)": f"{va.mean():.2f} ± {va.std(ddof=1):.2f}",
        "Test Acc (%)": f"{te.mean():.2f} ± {te.std(ddof=1):.2f}",
        "Best Test (%)": f"{te.max():.2f}",
        "Worst Test (%)": f"{te.min():.2f}",
        "Best epoch": f"{np.mean([r['best_epoch'] for r in rs]):.0f}",
        "Train time (s)": f"{np.mean([r['train_time'] for r in rs]):.2f}",
    })
results_df = pd.DataFrame(rows).set_index("Model")
results_df

### 4.1 Statistical significance
Seeds are shared between the two models, so we use a **paired t-test** and a Wilcoxon signed-rank test, plus Cohen's $d_z$ for the effect size.

In [ ]:
base = np.array([r["test_acc"] for r in runs["Baseline: 2-layer GCN"]]) * 100
prop = np.array([r["test_acc"] for r in runs["Proposed: 3-layer GCN"]]) * 100
diff = prop - base

t_stat, p_t = stats.ttest_rel(prop, base)
try:
    w_stat, p_w = stats.wilcoxon(prop, base)
except ValueError:                       # all differences zero
    w_stat, p_w = np.nan, 1.0
d_z = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else 0.0
ci = stats.t.interval(0.95, len(diff)-1, loc=diff.mean(), scale=stats.sem(diff))

print(f"Mean difference (3L - 2L): {diff.mean():+.2f} pp   95% CI [{ci[0]:+.2f}, {ci[1]:+.2f}]")
print(f"Paired t-test : t = {t_stat:.3f}, p = {p_t:.4f}")
print(f"Wilcoxon      : W = {w_stat}, p = {p_w:.4f}")
print(f"Cohen's d_z   : {d_z:.3f}")
print("Significant at 5%?", "YES" if p_t < 0.05 else "NO")
pd.DataFrame({"seed": SEEDS, "2-layer": base, "3-layer": prop, "diff": diff}).round(2)

### 4.2 Training curves

In [ ]:
def mean_curve(rs, key): return np.mean([r["hist"][key] for r in rs], axis=0)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
colors = {"Baseline: 2-layer GCN": "tab:blue", "Proposed: 3-layer GCN": "tab:red"}
for name, rs in runs.items():
    c = colors[name]
    ax[0].plot(mean_curve(rs, "train_loss"), c=c, label=name)
    ax[1].plot(mean_curve(rs, "val_loss"),   c=c, label=name)
    ax[2].plot(mean_curve(rs, "val_acc"),    c=c, label=name)
ax[0].set_title("Training loss (mean of 10 seeds)")
ax[1].set_title("Validation loss"); ax[2].set_title("Validation accuracy")
for a in ax: a.set_xlabel("Epoch"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()

### 4.3 Per-class and confusion analysis (seed 0)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, (name, rs) in zip(ax, runs.items()):
    m = rs[0]["model"].eval()
    with torch.no_grad(): pred = m(data.x, A_hat).argmax(1)[data.test_mask].cpu().numpy()
    true = data.y[data.test_mask].cpu().numpy()
    cm = confusion_matrix(true, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=a)
    a.set_title(f"{name}\nTest acc {np.mean(pred==true)*100:.1f}%"); a.set_xlabel("Predicted"); a.set_ylabel("True")
plt.tight_layout(); plt.savefig("confusion.png", dpi=150); plt.show()

for name, rs in runs.items():
    m = rs[0]["model"].eval()
    with torch.no_grad(): pred = m(data.x, A_hat).argmax(1)[data.test_mask].cpu().numpy()
    print("=====", name); print(classification_report(data.y[data.test_mask].cpu().numpy(), pred, digits=3))

## 5. Depth Ablation (1 to 6 layers) and Over-Smoothing Analysis
To explain *why* the three-layer model behaves as it does, we vary depth while keeping everything else fixed (5 seeds per depth). For each trained model we also compute the **mean pairwise cosine similarity** of the last hidden layer's node embeddings (on 1,500 random nodes). Values near 1 indicate over-smoothing.

In [ ]:
@torch.no_grad()
def smoothness(model, n_sample=1500, seed=0):
    model.eval()
    _, hidden = model(data.x, A_hat, return_hidden=True)
    if not hidden: return np.nan
    h = hidden[-1]
    g = torch.Generator(device="cpu").manual_seed(seed)
    idx = torch.randperm(h.size(0), generator=g)[:n_sample].to(h.device)
    hn = F.normalize(h[idx], dim=1)
    sim = hn @ hn.T
    n = sim.size(0)
    return ((sim.sum() - sim.diag().sum()) / (n * (n - 1))).item()

DEPTHS, ABL_SEEDS = [1, 2, 3, 4, 5, 6], list(range(5))
abl = []
for L in DEPTHS:
    rs = [train_model(L, s) for s in ABL_SEEDS]
    abl.append(dict(layers=L,
                    test_mean=np.mean([r["test_acc"] for r in rs])*100,
                    test_std=np.std([r["test_acc"] for r in rs], ddof=1)*100,
                    train_acc=np.mean([r["hist"]["train_acc"][-1] for r in rs])*100,
                    cos_sim=np.nanmean([smoothness(r["model"]) for r in rs]),
                    params=rs[0]["params"],
                    time=np.mean([r["train_time"] for r in rs])))
abl_df = pd.DataFrame(abl).set_index("layers").round(3)
abl_df

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].errorbar(abl_df.index, abl_df.test_mean, yerr=abl_df.test_std, marker="o", capsize=4)
ax[0].axvline(2, ls="--", c="tab:blue", alpha=.5); ax[0].axvline(3, ls="--", c="tab:red", alpha=.5)
ax[0].set_title("Test accuracy vs. depth"); ax[0].set_ylabel("Test acc (%)")
ax[1].plot(abl_df.index, abl_df.cos_sim, marker="s", c="tab:green")
ax[1].set_title("Over-smoothing: mean pairwise cosine sim."); ax[1].set_ylabel("Cosine similarity")
ax[2].plot(abl_df.index, abl_df.train_acc, marker="^", c="tab:purple")
ax[2].set_title("Final training accuracy (%)")
for a in ax: a.set_xlabel("Number of GCN layers"); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("depth_ablation.png", dpi=150); plt.show()

## 6. Embedding Visualization (t-SNE)
t-SNE of the last hidden layer (test nodes) for both models (seed 0).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, (name, rs) in zip(ax, runs.items()):
    m = rs[0]["model"].eval()
    with torch.no_grad(): _, hid = m(data.x, A_hat, return_hidden=True)
    emb = hid[-1][data.test_mask].cpu().numpy()
    z = TSNE(n_components=2, random_state=0, init="pca", perplexity=30).fit_transform(emb)
    sc = a.scatter(z[:, 0], z[:, 1], c=data.y[data.test_mask].cpu().numpy(), cmap="tab10", s=10)
    a.set_title(f"{name}\n(test acc {rs[0]['test_acc']*100:.1f}%)"); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig("tsne.png", dpi=150); plt.show()

## 7. Auto-generated Result Summary
Run this cell and copy the printed numbers into your report.

In [ ]:
b_m, b_s = base.mean(), base.std(ddof=1)
p_m, p_s = prop.mean(), prop.std(ddof=1)
winner = "3-layer (proposed)" if p_m > b_m else "2-layer (baseline)"
print(f"""
RESULT SUMMARY (Cora, Planetoid split, {len(SEEDS)} seeds)
-----------------------------------------------------------
Baseline 2-layer GCN : {b_m:.2f} ± {b_s:.2f} % test accuracy  ({runs['Baseline: 2-layer GCN'][0]['params']:,} params)
Proposed 3-layer GCN : {p_m:.2f} ± {p_s:.2f} % test accuracy  ({runs['Proposed: 3-layer GCN'][0]['params']:,} params)
Difference           : {p_m-b_m:+.2f} pp (95% CI [{ci[0]:+.2f}, {ci[1]:+.2f}])
Paired t-test p      : {p_t:.4f}   | Wilcoxon p: {p_w:.4f}   | Cohen's d_z: {d_z:.2f}
Higher mean accuracy : {winner}
Statistically significant (p<0.05): {'YES' if p_t < 0.05 else 'NO'}
Best depth in ablation: {int(abl_df.test_mean.idxmax())} layers ({abl_df.test_mean.max():.2f} %)
""")
results_df.to_csv("main_results.csv"); abl_df.to_csv("depth_ablation.csv")
print("Saved: main_results.csv, depth_ablation.csv, training_curves.png, confusion.png, depth_ablation.png, tsne.png")

## 8. Discussion (fill in with your numbers)

**8.1 Accuracy.** Report the Section 7 summary: mean ± std test accuracy for both models, the difference, and the p-value. State plainly whether the 3-layer GCN improves, matches, or underperforms the baseline. A statistically insignificant difference is still a valid, publishable finding.

**8.2 Why depth behaves this way (link back to the derivation).**
- *Receptive field:* 3 layers aggregate $\hat A^3$-hop information (Section 1.7), which can help low-degree nodes far from labelled nodes.
- *Over-smoothing:* each extra $\hat A$ multiplication contracts feature differences (Section 1.9). Check the cosine-similarity curve in Section 5: if it rises noticeably from 2 to 3 layers, smoothing offsets the benefit of the larger receptive field.
- *Optimization/overfitting:* only 140 labelled nodes and $h^2+h=272$ extra parameters. Compare training accuracy and validation loss curves (Section 4.2) for signs of overfitting or slower convergence.

**8.3 Cost.** The proposed model adds one sparse multiplication per forward pass, so training time grows roughly linearly with depth (see the `Train time` column).

**8.4 Limitations and future work.**
- Single dataset (extend to Citeseer / Pubmed with `name="CiteSeer"` / `"PubMed"`).
- Fixed hyperparameters (hidden size, dropout, weight decay could be tuned separately for each depth).
- Possible extensions that stay close to Vanilla GCN: residual connections, PairNorm / DropEdge, or a hidden-size sweep.

**References**
1. T. N. Kipf and M. Welling, *Semi-Supervised Classification with Graph Convolutional Networks*, ICLR 2017.
2. M. Defferrard, X. Bresson, P. Vandergheynst, *Convolutional Neural Networks on Graphs with Fast Localized Spectral Filtering*, NeurIPS 2016.
3. Q. Li, Z. Han, X.-M. Wu, *Deeper Insights into Graph Convolutional Networks for Semi-Supervised Learning*, AAAI 2018.
4. D. K. Hammond, P. Vandergheynst, R. Gribonval, *Wavelets on Graphs via Spectral Graph Theory*, 2011.